## Week-2

In [2]:
import pandas as pd

df = pd.read_csv("CRMLSSold_residential.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_3332/1797885786.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSSold_residential.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,94401,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,...,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
4,Southland,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,karen.klein@theagencyre.com,...,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN


In [3]:
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(exclude=["number"]).columns

print("Numerical columns:", len(num_cols))
print(num_cols)

Numerical columns: 32
Index(['OriginalListPrice', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'FireplacesTotal', 'AboveGradeFinishedArea', 'ListingKeyNumeric',
       'TaxAnnualAmount', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt',
       'StreetNumberNumeric', 'BathroomsTotalInteger', 'TaxYear',
       'BuildingAreaTotal', 'BedroomsTotal', 'ElementarySchoolDistrict',
       'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'Stories',
       'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee',
       'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict',
       'BuyerAgencyCompensation'],
      dtype='str')


In [4]:
import pandas as pd
import numpy as np
from IPython.display import display

# =========================
# 0️⃣ Duplicate 
# =========================
total_duplicates = df.duplicated().sum()

if "ListingId" in df.columns:
    listing_duplicates = df.duplicated(subset=["ListingId"]).sum()
else:
    listing_duplicates = None

print("=== DUPLICATE CHECK ===")
print("Total duplicate rows:", total_duplicates)
print("Duplicate ListingId:", listing_duplicates)


# =========================
num_keep_cols = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
    'DaysOnMarket', 'YearBuilt',
    'LotSizeSquareFeet',
    'Latitude', 'Longitude',
    'ParkingTotal', 'GarageSpaces',
    'Stories',
    'month'
]

# 
num_keep_cols = [
    col for col in num_keep_cols
    if col in df.columns and df[col].isna().mean() < 1
]


# =========================
# 
# =========================
num_cols = df.select_dtypes(include=np.number).columns.tolist()
summary = df[num_cols].describe().T

report = []

for col in num_cols:
    col_data = df[col].dropna()

    missing_count = df[col].isna().sum()
    missing_pct = round(df[col].isna().mean() * 100, 2)

    if col_data.empty:
        report.append([col, False, missing_count, missing_pct, 0, 0,
                       np.nan, np.nan, np.nan, np.nan,
                       np.nan, np.nan, 0, False, False, "Drop"])
        continue

    # 
    min_val = summary.loc[col, 'min']
    max_val = summary.loc[col, 'max']
    std_val = summary.loc[col, 'std']
    mean_val = summary.loc[col, 'mean']

    # 
    negative_count = int((col_data < 0).sum())
    zero_count = int((col_data == 0).sum())

    # extreme max
    p99 = col_data.quantile(0.99)
    extreme_max = max_val > p99 * 5 if pd.notna(p99) and p99 != 0 else False

    # high std
    high_std = std_val > abs(mean_val) * 2 if pd.notna(mean_val) and mean_val != 0 else False

    # IQR
    q1 = col_data.quantile(0.25)
    q3 = col_data.quantile(0.75)
    iqr = q3 - q1

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = int(((col_data < lower) | (col_data > upper)).sum())
    else:
        outlier_count = 0

    # 
    is_core = col in num_keep_cols

    report.append([
        col, is_core,
        missing_count, missing_pct,
        negative_count, zero_count,
        round(min_val, 2), round(max_val, 2),
        round(mean_val, 2), round(std_val, 2),
        round(q1, 2), round(q3, 2),
        outlier_count, extreme_max, high_std,
        None
    ])


# =========================
# 3️⃣ DataFrame
# =========================
report_df = pd.DataFrame(report, columns=[
    'Column', 'Is Core',
    'Missing', 'Missing %',
    'Negative Count', 'Zero Count',
    'Min', 'Max', 'Mean', 'Std',
    'Q1', 'Q3',
    'IQR Outliers',
    'Extreme Max ⚠️',
    'High Std ⚠️',
    'Action'
])


# =========================
# 4️⃣ Action ）
# =========================
def assign_action(row):
    if row["Missing %"] > 90:
        return "Drop"

    if row["Is Core"]:
        if row["Negative Count"] > 0:
            return "Fix Negative"
        elif row["Extreme Max ⚠️"] or row["High Std ⚠️"] or row["IQR Outliers"] > 0:
            return "Cap / Clean"
        else:
            return "OK"
    else:
        if row["Negative Count"] > 0:
            return "Fix Negative"
        elif row["Extreme Max ⚠️"] or row["High Std ⚠️"] or row["IQR Outliers"] > 0:
            return "Cap / Clean"
        else:
            return "OK"

report_df["Action"] = report_df.apply(assign_action, axis=1)


# =========================
# 5️⃣ ）
# =========================
action_order = {
    "Drop": 0,
    "Cap / Clean": 1,
    "Fix Negative": 2,
    "OK": 3
}

report_df["ActionRank"] = report_df["Action"].map(action_order)

report_df = report_df.sort_values(
    by=["ActionRank", "Missing %"],
    ascending=[True, False]
).reset_index(drop=True)


# =========================
# 6️⃣ 
# =========================


print("=== DROP ===")
display(report_df[report_df["Action"] == "Drop"])


print("=== CAP / CLEAN ===")
display(report_df[report_df["Action"] == "Cap / Clean"])


print("=== KEEP (OK + Fix Negative) ===")
display(report_df[report_df["Action"].isin(["OK", "Fix Negative"])])


print("=== KEEP (CORE) ===")
display(
    report_df[report_df["Is Core"]]
    .sort_values(by=["ActionRank", "Missing %"])
)

=== DUPLICATE CHECK ===
Total duplicate rows: 48
Duplicate ListingId: 330
=== DROP ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
0,FireplacesTotal,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
1,AboveGradeFinishedArea,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
2,TaxAnnualAmount,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
3,TaxYear,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
4,ElementarySchoolDistrict,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
5,BusinessType,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
6,CoveredSpaces,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
7,MiddleOrJuniorSchoolDistrict,False,397603,100.00,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,False,False,Drop,0
8,BelowGradeFinishedArea,False,395313,99.42,0,1890,0.0,14531.0,98.00,737.04,0.0,0.0,0,True,True,Drop,0
9,BuildingAreaTotal,False,369810,93.01,0,96,0.0,123764.0,2118.51,1704.70,1270.0,2458.0,1786,True,False,Drop,0


=== CAP / CLEAN ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
10,BuyerAgencyCompensation,False,351478,88.40,0,97,0.0,1.000000e+05,2.219300e+02,2590.41,2.000000e+00,2.500000e+00,1025,True,True,Cap / Clean,1
11,MainLevelBedrooms,False,166841,41.96,0,38579,0.0,3.333000e+03,2.060000e+00,7.24,1.000000e+00,3.000000e+00,431,True,True,Cap / Clean,1
12,AssociationFee,False,90998,22.89,0,138554,0.0,7.500000e+05,2.348500e+02,1772.30,0.000000e+00,3.690000e+02,8738,True,True,Cap / Clean,1
13,LotSizeAcres,False,31364,7.89,0,8374,0.0,7.810698e+06,6.856000e+01,16327.24,1.200000e-01,2.700000e-01,56982,True,True,Cap / Clean,1
14,LotSizeSquareFeet,True,30987,7.79,0,8097,0.0,2.208318e+09,3.347135e+05,16306717.36,5.227000e+03,1.182500e+04,57591,True,True,Cap / Clean,1
15,LotSizeArea,False,30863,7.76,0,8109,0.0,9.187423e+08,3.839295e+04,2139732.14,5.000000e+03,1.089000e+04,55564,True,True,Cap / Clean,1
16,GarageSpaces,True,17053,4.29,0,44254,0.0,6.000000e+02,1.860000e+00,3.71,2.000000e+00,2.000000e+00,0,True,False,Cap / Clean,1
17,OriginalListPrice,True,721,0.18,0,2,0.0,1.390000e+09,1.224737e+06,6779803.84,5.850000e+05,1.299000e+06,30860,True,True,Cap / Clean,1
18,StreetNumberNumeric,False,445,0.11,0,209,0.0,7.814782e+07,1.044118e+04,228802.27,1.006000e+03,1.202775e+04,35030,True,True,Cap / Clean,1
19,YearBuilt,True,356,0.09,0,0,1776.0,2.026000e+03,1.978570e+03,26.27,1.960000e+03,1.999000e+03,942,False,False,Cap / Clean,1


=== KEEP (OK + Fix Negative) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
27,Latitude,True,15822,3.98,2,25,-117.47,56.13,34.65,1.71,33.74,34.55,77783,False,False,Fix Negative,2
28,Longitude,True,15822,3.98,381727,25,-177.65,329.00,-118.57,2.96,-118.88,-117.27,70276,True,False,Fix Negative,2
29,ParkingTotal,True,426,0.11,97,26482,-143.00,2593669.00,9.33,4115.66,2.00,3.00,59668,True,True,Fix Negative,2
30,DaysOnMarket,True,0,0.00,46,15809,-288.00,12430.00,37.34,53.54,8.00,48.00,30269,True,False,Fix Negative,2
31,Stories,True,61503,15.47,0,0,1.00,2.00,1.36,0.48,1.00,2.00,0,False,False,OK,3


=== KEEP (CORE) ===


,Column,Is Core,Missing,Missing %,Negative Count,Zero Count,Min,Max,Mean,Std,Q1,Q3,IQR Outliers,Extreme Max ⚠️,High Std ⚠️,Action,ActionRank
23,ClosePrice,True,2,0.00,0,1,0.00,9.895000e+08,1185616.36,5922380.41,575000.00,1300000.00,29410,True,True,Cap / Clean,1
24,ListPrice,True,0,0.00,0,0,525.00,1.375000e+08,1138630.30,1353707.09,575000.00,1295000.00,29578,True,False,Cap / Clean,1
26,BedroomsTotal,True,11,0.00,0,1138,0.00,4.500000e+01,3.20,1.07,3.00,4.00,21913,True,False,Cap / Clean,1
21,BathroomsTotalInteger,True,69,0.02,0,305,0.00,1.750000e+02,2.53,1.14,2.00,3.00,18245,True,False,Cap / Clean,1
20,LivingArea,True,229,0.06,0,144,0.00,1.702132e+07,1904.35,27017.81,1247.00,2217.00,17540,True,True,Cap / Clean,1
19,YearBuilt,True,356,0.09,0,0,1776.00,2.026000e+03,1978.57,26.27,1960.00,1999.00,942,False,False,Cap / Clean,1
17,OriginalListPrice,True,721,0.18,0,2,0.00,1.390000e+09,1224736.70,6779803.84,585000.00,1299000.00,30860,True,True,Cap / Clean,1
16,GarageSpaces,True,17053,4.29,0,44254,0.00,6.000000e+02,1.86,3.71,2.00,2.00,0,True,False,Cap / Clean,1
14,LotSizeSquareFeet,True,30987,7.79,0,8097,0.00,2.208318e+09,334713.47,16306717.36,5227.00,11825.00,57591,True,True,Cap / Clean,1
30,DaysOnMarket,True,0,0.00,46,15809,-288.00,1.243000e+04,37.34,53.54,8.00,48.00,30269,True,False,Fix Negative,2
